# 02 — Parsing

DocuFlow's 4 parsers and how they extract text + bounding boxes from documents.

## From pixels to structured text

Before an LLM can extract data from a document, something needs to convert the raw file into text. This is the parser's job, and it is more important than it looks — the quality of parsing directly determines the quality of extraction.

A parser produces three things:
1. **Text blocks** — chunks of text found on each page, in reading order.
2. **Bounding boxes** — the exact `(x0, y0, x1, y1)` coordinates of each block on the page, in PDF points (1 point = 1/72 inch). These are the foundation for evidence grounding: when DocuFlow says a value was found at a specific location, it is because the parser placed a block there.
3. **Page metadata** — dimensions, block count, and the full raw text for each page.

The challenge is that PDFs come in two fundamentally different forms:

- **Digital (native) PDFs** have an embedded text layer. Parsers like pdfplumber can extract text directly from the file structure in milliseconds, with pixel-perfect bounding boxes.
- **Scanned PDFs** are images wrapped in a PDF container — there is no text layer. These require OCR (optical character recognition) to recover the text. OCR is slower, less accurate, and the bounding boxes are approximate.

The **Smart parser** solves the "I don't know what kind of PDF this is" problem: it tries pdfplumber first, checks if each page has enough usable text, and falls back to Tesseract OCR for pages that don't. This per-page decision means a document with some digital pages and some scanned pages gets the best parser for each.

In [1]:
PDF_PATH = "data/sample_invoice.pdf"

## Parser Overview

DocuFlow ships 4 parsers. All produce structured `Page` and `Block` objects with bounding boxes.

| Parser | Best for | Speed |
|---|---|---|
| `PdfplumberParser` | Digital / native PDFs | ~100 ms/page |
| `TesseractParser` | Scanned / image-based PDFs | 1-5 s/page |
| `DoclingParser` | Complex tables and layouts | 4-5 s/page |
| `SmartParser` | Mixed documents (auto per-page) | Varies |

The smart parser tries pdfplumber first, then falls back to Tesseract OCR for pages that lack usable text.

## Cloud OCR parsers

Three managed OCR services plug in as parsers and produce the same standardized output (line-level blocks with per-word confidences):

- `parser="azure-di"` — Azure Document Intelligence (`pip install docuflow[azure]`)
- `parser="textract"` — AWS Textract, no S3 bucket needed (`pip install docuflow[aws]`)
- `parser="google-docai"` — Google Document AI (`pip install docuflow[gcp]`)

```python
pipeline = DocumentPipeline(parser={"type": "azure-di", "model": "prebuilt-read"})
```

Credentials come from environment variables (see the guide). All parsers emit bounding boxes in one canonical coordinate space — top-left origin, PDF points — so `bbox.to_relative(page.width, page.height)` gives overlay-ready 0-1 coordinates for any parser at any render DPI.

## pdfplumber Parser

The fastest parser -- ideal for digital PDFs where text is embedded (not scanned).

In [2]:
from docuflow.parsing.pdfplumber_parser import PdfplumberParser
from docuflow.workflow.pipeline import Pipeline
from docuflow.workflow.steps import Ingest, Parse

parser = PdfplumberParser()
pipeline = Pipeline([Ingest(path=PDF_PATH), Parse(parser=parser)])
result = pipeline.run_sync()
document = result.state.document

print(f"Pages        : {len(document.pages)}")
print(f"Total blocks : {sum(p.block_count for p in document.pages)}")
print(f"Status       : {document.status}")
print()
print("--- raw_text preview (first 500 chars) ---")
print(document.raw_text[:500])

Pages        : 2
Total blocks : 145
Status       : parsed

--- raw_text preview (first 500 chars) ---
INVOICE
Meridian Dynamics Ltd
#INV-2024-1847
From: Invoice Date: November 15, 2024
Meridian Dynamics Ltd Due Date: December 15, 2024
47 Innovation Drive, Suite 300 PO Number: PO-90281
Cambridge, MA 02142
Payment Terms: Net 30
United States
Tax ID: 84-2917305
Phone: +1 (617) 555-0194
Email: billing@meridiandynamics.com
BILL TO SHIP TO
Northwind Traders Inc. Northwind Traders — Warehouse B
Attn: Sarah Chen, Procurement Manager Attn: James Rodriguez, Receiving
1200 Market Street, Floor 8 800 Indust


## Exploring Pages and Blocks

Each page contains a list of `Block` objects. Every block has a `text`, `block_type`, and an optional `BoundingBox` with coordinates on the page.

In [3]:
for page in document.pages:
    print(f"=== Page {page.page_number} === ({page.block_count} blocks, {page.width:.0f}x{page.height:.0f} pts)")
    for i, block in enumerate(page.blocks[:5]):
        text_preview = block.text[:60].replace("\n", " ")
        print(f"  Block {i}: [{block.block_type.value}] \"{text_preview}\"")
        if block.bbox:
            bb = block.bbox
            print(f"           bbox: x0={bb.x0:.1f}, y0={bb.y0:.1f}, x1={bb.x1:.1f}, y1={bb.y1:.1f}"
                  f"  ({bb.width:.1f}w x {bb.height:.1f}h)")
    if page.block_count > 5:
        print(f"  ... and {page.block_count - 5} more blocks")
    print()

=== Page 0 === (97 blocks, 612x792 pts)
  Block 0: [text] "INVOICE"
           bbox: x0=400.0, y0=11.0, x1=498.7, y1=35.0  (98.7w x 24.0h)
  Block 1: [text] "Meridian Dynamics Ltd"
           bbox: x0=40.0, y0=29.1, x1=243.4, y1=49.1  (203.4w x 20.0h)
  Block 2: [text] "#INV-2024-1847"
           bbox: x0=400.0, y0=41.3, x1=480.7, y1=52.3  (80.7w x 11.0h)
  Block 3: [text] "From:"
           bbox: x0=40.0, y0=88.7, x1=60.9, y1=96.7  (20.9w x 8.0h)
  Block 4: [text] "Invoice Date:"
           bbox: x0=380.0, y0=88.7, x1=426.7, y1=96.7  (46.7w x 8.0h)
  ... and 92 more blocks

=== Page 1 === (48 blocks, 612x792 pts)
  Block 0: [text] "Invoice #INV-2024-1847 — continued"
           bbox: x0=40.0, y0=32.1, x1=206.8, y1=42.1  (166.8w x 10.0h)
  Block 1: [text] "# Description"
           bbox: x0=40.0, y0=65.7, x1=95.0, y1=73.7  (55.0w x 8.0h)
  Block 2: [text] "Qty"
           bbox: x0=280.0, y0=65.7, x1=292.4, y1=73.7  (12.4w x 8.0h)
  Block 3: [text] "Unit Price"
           bbox: x0=340.0

## Block Types and Structure

Blocks are classified by type (text, image, title, etc.). Below is the distribution across the document.

In [4]:
from collections import Counter

all_blocks = [b for p in document.pages for b in p.blocks]
type_counts = Counter(b.block_type.value for b in all_blocks)
text_lengths = [len(b.text) for b in all_blocks if b.text]

print("Block type distribution:")
max_count = max(type_counts.values())
for btype, count in type_counts.most_common():
    bar = "#" * int(count / max_count * 40)
    print(f"  {btype:<12} {count:>3}  {bar}")

print()
print(f"Text length stats across {len(text_lengths)} blocks:")
print(f"  Shortest : {min(text_lengths):>5} chars")
print(f"  Longest  : {max(text_lengths):>5} chars")
print(f"  Average  : {sum(text_lengths) / len(text_lengths):>5.0f} chars")
print(f"  Total    : {sum(text_lengths):>5} chars")

Block type distribution:
  text         145  ########################################

Text length stats across 145 blocks:
  Shortest :     1 chars
  Longest  :    91 chars
  Average  :    16 chars
  Total    :  2276 chars


## Tesseract Parser

OCR-based parser for scanned or image-based PDFs. Each page is rendered as an image, then Tesseract extracts text as line-level blocks, each carrying per-word bounding boxes and confidence scores (the words feed the OCR confidence scores and word-precise evidence highlighting). Slower than pdfplumber but essential when there is no embedded text layer.

In [5]:
from docuflow.parsing.tesseract_parser import TesseractParser

tesseract = TesseractParser()
pipeline_tess = Pipeline([Ingest(path=PDF_PATH), Parse(parser=tesseract)])
result_tess = pipeline_tess.run_sync()
doc_tess = result_tess.state.document

tess_blocks = sum(p.block_count for p in doc_tess.pages)

print(f"Pages        : {len(doc_tess.pages)}")
print(f"Total blocks : {tess_blocks}")
print()
print(f"Comparison: pdfplumber produced {sum(p.block_count for p in document.pages)} blocks, "
      f"Tesseract produced {tess_blocks} blocks")
print()
print("--- Tesseract raw_text preview (first 500 chars) ---")
print(doc_tess.raw_text[:500])

Pages        : 2
Total blocks : 51

Comparison: pdfplumber produced 145 blocks, Tesseract produced 51 blocks

--- Tesseract raw_text preview (first 500 chars) ---
Meridian Dynamics Ltd 4INV-2024-1847
From: Invoice Date: November 15, 2024
Meridian Dynamics Ltd Due Date: December 15, 2024
47 Innovation Drive, Suite 300 PO Number: PO-90281
Cambridge, MA 02142 Payment Terms: Net 30
United States
Tax ID: 84-2917305
Phone: +1 (617) 555-0194
Email: billing@meridiandynamics.com

BILL TO SHIP TO

Northwind Traders Inc. Northwind Traders — Warehouse B

Attn: Sarah Chen, Procurement Manager Attn: James Rodriguez, Receiving

1200 Market Street, Floor 8 800 Industrial


## Docling Parser

The highest-quality parser for documents with complex tables, multi-column layouts, and nested structures. Docling understands document structure at a semantic level -- it produces first-class `Table` objects with row/column headers and per-cell bounding boxes. Significantly slower than pdfplumber but worth it when table extraction matters.

> **Note:** Docling requires `pip install docuflow[docling]` and has heavy dependencies. The cell below is wrapped in a try/except so the notebook runs even if Docling is not installed.

In [6]:
try:
    from docuflow.parsing.docling_parser import DoclingParser

    docling = DoclingParser()
    pipeline_doc = Pipeline([Ingest(path=PDF_PATH), Parse(parser=docling)])
    result_doc = pipeline_doc.run_sync()
    doc_docling = result_doc.state.document

    docling_blocks = sum(p.block_count for p in doc_docling.pages)

    print(f"Pages        : {len(doc_docling.pages)}")
    print(f"Total blocks : {docling_blocks}")
    print()

    for page in doc_docling.pages:
        tables = page.tables if hasattr(page, "tables") else []
        print(f"Page {page.page_number}: {page.block_count} blocks, {len(tables)} table(s)")
        for table in tables:
            print(f"  Table: {table.row_count} rows x {table.col_count} cols")
            if table.row_count > 0 and table.col_count > 0:
                cell = table.cell_at(0, 0)
                print(f"    cell(0,0): \"{cell.text}\"")

    print()
    print(f"Comparison: pdfplumber={sum(p.block_count for p in document.pages)}, "
          f"Tesseract={tess_blocks}, Docling={docling_blocks} blocks")

except ImportError:
    print("Docling is not installed. Install with: pip install docuflow[docling]")
    print("Skipping Docling parser demo.")

/Users/valerio/Repos/DocFlow/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-06-13 10:25:17,495 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-13 10:25:17,498 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-13 10:25:17,510 [RapidOCR] download_file.py:60: File exists and is valid: /Users/valerio/Repos/DocFlow/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-13 10:25:17,510 [RapidOCR] main.py:50: Using /Users/valerio/Repos/DocFlow/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-13 10:25:17,629 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-13 10:25:17,630 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-13 10:25:17,631 [RapidOCR] download

Pages        : 0
Total blocks : 0


Comparison: pdfplumber=145, Tesseract=51, Docling=0 blocks


## Smart Parser

The smart parser runs pdfplumber first, then checks each page for usable text. Pages with too little text (scanned, garbled, image-heavy) are re-processed with Tesseract OCR.

In [7]:
from docuflow.parsing.smart_parser import SmartParser

smart = SmartParser()
pipeline2 = Pipeline([Ingest(path=PDF_PATH), Parse(parser=smart)])
result2 = pipeline2.run_sync()
doc2 = result2.state.document

smart_blocks = sum(p.block_count for p in doc2.pages)
pdfplumber_blocks = sum(p.block_count for p in document.pages)

print(f"pdfplumber blocks : {pdfplumber_blocks}")
print(f"Smart blocks   : {smart_blocks}")
print()
if smart_blocks == pdfplumber_blocks:
    print("Block counts match -- the smart parser detected this PDF as fully digital")
    print("and used pdfplumber for all pages (no OCR fallback needed).")
else:
    print(f"Difference: {abs(smart_blocks - pdfplumber_blocks)} blocks.")
    print("The smart parser used OCR for some pages.")

pdfplumber blocks : 145
Smart blocks   : 145

Block counts match -- the smart parser detected this PDF as fully digital
and used pdfplumber for all pages (no OCR fallback needed).


## Searching Within Parsed Documents

`search_document` finds text across all pages and returns hits with page numbers, bounding boxes, and surrounding context.

In [8]:
from docuflow.search import search_document

hits = search_document(document, "Meridian")

print(f"Query: \"Meridian\"")
print(f"Total hits: {hits.total_hits}")
print()
for i, hit in enumerate(hits.hits):
    print(f"  Hit {i + 1}:")
    print(f"    Page     : {hit.page_number}")
    print(f"    Text     : \"{hit.text}\"")
    print(f"    Context  : \"{hit.context}\"")
    if hit.bbox:
        bb = hit.bbox
        print(f"    BBox     : x0={bb.x0:.1f}, y0={bb.y0:.1f}, x1={bb.x1:.1f}, y1={bb.y1:.1f}")
    print()

Query: "Meridian"
Total hits: 6

  Hit 1:
    Page     : 0
    Text     : "Meridian"
    Context  : "INVOICE
Meridian Dynamics Ltd
#INV-2024-1847
From: Invoice Date: N..."
    BBox     : x0=40.0, y0=29.1, x1=116.7, y1=49.1

  Hit 2:
    Page     : 0
    Text     : "Meridian"
    Context  : "INVOICE
Meridian Dynamics Ltd
#INV-2024-1847
From: Invoice Date: N..."
    BBox     : x0=40.0, y0=101.9, x1=74.5, y1=110.9

  Hit 3:
    Page     : 0
    Text     : "billing@meridiandynamics.com"
    Context  : "...ax ID: 84-2917305
Phone: +1 (617) 555-0194
Email: billing@meridiandynamics.com
BILL TO SHIP TO
Northwind Traders Inc. Northwind ..."
    BBox     : x0=64.4, y0=169.7, x1=174.8, y1=177.7

  Hit 4:
    Page     : 1
    Text     : "Meridian"
    Context  : "...k: First National Bank of Cambridge
Account Name: Meridian Dynamics Ltd
Routing Number: 021000089
Account Nu..."
    BBox     : x0=160.0, y0=269.7, x1=190.7, y1=277.7

  Hit 5:
    Page     : 1
    Text     : "Meridian"
    Context  : 

## Screenshots

Render PDF pages as PNG images. Useful for review UIs or visual debugging of bounding boxes.

In [9]:
from docuflow.screenshots import screenshot_pages_sync

shots = screenshot_pages_sync(file_path=PDF_PATH, output_dir="./output/pages", dpi=150)
print(f"Rendered {len(shots)} page(s)")
for s in shots:
    print(f"  Page {s.page_number}: {s.width}x{s.height} px  ->  {s.file_path}")

Rendered 2 page(s)
  Page 0: 1275x1651 px  ->  output/pages/sample_invoice_page_0.png
  Page 1: 1275x1651 px  ->  output/pages/sample_invoice_page_1.png


## Highlighted renders

After parsing, you can run extraction and then render each page with colored bounding boxes showing exactly where each field was found. `highlight_fields` saves one annotated PNG per page and returns the paths.

In [ ]:
from docuflow import extract, highlight_fields
from pydantic import BaseModel, Field

# Quick extraction so we have bounding boxes to highlight
class InvoiceSnippet(BaseModel):
    supplier_name: str = Field(description="Supplier name")
    invoice_number: str = Field(description="Invoice number")
    total: float = Field(description="Total amount")

result = extract(PDF_PATH, schema=InvoiceSnippet)

# Render with highlights — yellow by default
saved = highlight_fields(PDF_PATH, result, output_dir="/tmp/docuflow_parsing")

from IPython.display import Image as IPImage, display
for p in saved:
    print("Saved:", p)
    display(IPImage(p, width=700))

## Summary

Use **pdfplumber** for digital PDFs where speed matters, **Tesseract** for scanned documents, and **Docling** when you need structured table extraction. When you are not sure what kind of PDF you have, use **SmartParser** -- it picks the right strategy per page automatically.